# Chapter 12: AI Cost Calculator

**Build a cost calculator for your AI workload. Model different scenarios — model tiering, caching strategies, batch vs. real-time — and see the dollar impact of each optimization.**

AI spend can spiral quickly. A single unoptimized endpoint doing 10K requests/day at
$15/M input tokens adds up fast. In this notebook, we build a `CostCalculator` that
models five scenarios and shows exactly where the savings come from.

**Key insight**: Systematic cost optimization can reduce AI spend by 50–90%.

---

## Setup

In [ ]:
# Install dependencies (run once)
!pip install -q tabulate

In [ ]:
import os
from dataclasses import dataclass, field
from typing import Dict
from tabulate import tabulate

# No API keys needed — this notebook is entirely local computation.
print("Setup complete. No API keys required for this notebook.")

## 1. Define Model Pricing

We define pricing for several popular models. Prices are per 1 million tokens (USD).
These represent approximate list prices — check provider docs for current rates.

In [ ]:
# Pricing per 1M tokens (USD)
MODEL_PRICING = {
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "claude-sonnet": {"input": 3.00, "output": 15.00},
    "claude-haiku": {"input": 0.25, "output": 1.25},
}

# Display the pricing table
pricing_rows = []
for model, prices in MODEL_PRICING.items():
    pricing_rows.append([model, f"${prices['input']:.2f}", f"${prices['output']:.2f}"])

print(tabulate(pricing_rows,
               headers=["Model", "Input ($/1M tokens)", "Output ($/1M tokens)"],
               tablefmt="github"))

## 2. Build the CostCalculator

The calculator takes a workload profile and computes costs under five scenarios:

| # | Scenario | What changes |
|---|----------|-------------|
| 1 | **Baseline** | All requests to a premium model, no optimizations |
| 2 | **+ Caching** | Cache hit rate reduces the number of LLM calls |
| 3 | **+ Tiering** | Route requests to cheaper models by complexity |
| 4 | **+ Batching** | Non-real-time requests get a batch discount |
| 5 | **All combined** | Caching + tiering + batching together |

In [ ]:
@dataclass
class CostCalculator:
    """Calculate and compare AI workload costs across optimization scenarios."""

    # ── Workload parameters ──────────────────────────────────────────
    requests_per_day: int = 10_000
    avg_input_tokens: int = 800
    avg_output_tokens: int = 400

    # ── Model configuration ──────────────────────────────────────────
    baseline_model: str = "gpt-4o"  # the "default" expensive model
    model_pricing: Dict = field(default_factory=lambda: MODEL_PRICING)

    # ── Optimization parameters ──────────────────────────────────────
    cache_hit_rate: float = 0.30        # 30% of requests served from cache
    batch_discount: float = 0.50        # 50% discount on batch requests
    batch_eligible_pct: float = 0.40    # 40% of requests can be batched

    # Tier distribution: what fraction of requests fall into each tier?
    tier_distribution: Dict = field(default_factory=lambda: {
        "simple": 0.50,    # 50% simple
        "medium": 0.35,    # 35% medium
        "complex": 0.15,   # 15% complex
    })

    # Which model handles each tier?
    tier_model_map: Dict = field(default_factory=lambda: {
        "simple": "gpt-4o-mini",
        "medium": "claude-haiku",
        "complex": "gpt-4o",
    })

    # ── Internal helpers ─────────────────────────────────────────────
    def _cost_per_request(self, model: str) -> float:
        """Cost of a single average request for the given model."""
        p = self.model_pricing[model]
        return (
            (self.avg_input_tokens / 1_000_000) * p["input"]
            + (self.avg_output_tokens / 1_000_000) * p["output"]
        )

    def _daily(self, cost_per_req: float, num_requests: int = None) -> float:
        """Daily cost."""
        n = num_requests if num_requests is not None else self.requests_per_day
        return cost_per_req * n

    # ── Scenario calculations ────────────────────────────────────────
    def scenario_baseline(self) -> float:
        """Scenario 1: All requests to the baseline (premium) model."""
        return self._daily(self._cost_per_request(self.baseline_model))

    def scenario_with_caching(self) -> float:
        """Scenario 2: Caching reduces the number of LLM calls."""
        effective_requests = self.requests_per_day * (1 - self.cache_hit_rate)
        return self._daily(self._cost_per_request(self.baseline_model),
                           int(effective_requests))

    def scenario_with_tiering(self) -> float:
        """Scenario 3: Route requests to different models by complexity tier."""
        daily_cost = 0
        for tier, fraction in self.tier_distribution.items():
            model = self.tier_model_map[tier]
            tier_requests = int(self.requests_per_day * fraction)
            daily_cost += self._daily(self._cost_per_request(model), tier_requests)
        return daily_cost

    def scenario_with_batching(self) -> float:
        """Scenario 4: Batch-eligible requests get a discount."""
        cpr = self._cost_per_request(self.baseline_model)
        realtime_requests = int(self.requests_per_day * (1 - self.batch_eligible_pct))
        batch_requests = int(self.requests_per_day * self.batch_eligible_pct)
        return (
            self._daily(cpr, realtime_requests)
            + self._daily(cpr * (1 - self.batch_discount), batch_requests)
        )

    def scenario_all_optimized(self) -> float:
        """Scenario 5: Caching + tiering + batching combined."""
        # First, caching reduces request volume
        effective_requests = self.requests_per_day * (1 - self.cache_hit_rate)

        daily_cost = 0
        for tier, fraction in self.tier_distribution.items():
            model = self.tier_model_map[tier]
            tier_requests = effective_requests * fraction

            # Split into real-time and batch-eligible
            realtime = tier_requests * (1 - self.batch_eligible_pct)
            batch = tier_requests * self.batch_eligible_pct

            cpr = self._cost_per_request(model)
            daily_cost += realtime * cpr
            daily_cost += batch * cpr * (1 - self.batch_discount)

        return daily_cost

    def compare_all(self) -> list:
        """Run all five scenarios and return comparison data."""
        baseline = self.scenario_baseline()
        scenarios = [
            ("1. Baseline (all premium)", self.scenario_baseline()),
            ("2. + Caching", self.scenario_with_caching()),
            ("3. + Tiering", self.scenario_with_tiering()),
            ("4. + Batching", self.scenario_with_batching()),
            ("5. All optimizations", self.scenario_all_optimized()),
        ]

        rows = []
        for name, daily_cost in scenarios:
            monthly = daily_cost * 30
            savings = baseline * 30 - monthly
            pct = (savings / (baseline * 30)) * 100 if baseline > 0 else 0
            rows.append([
                name,
                f"${daily_cost:,.2f}",
                f"${monthly:,.2f}",
                f"${savings:,.2f}" if savings > 0 else "--",
                f"{pct:.1f}%" if savings > 0 else "--",
            ])
        return rows


print("CostCalculator class defined.")

## 3. Run the Default Scenario

We use a workload of **10,000 requests/day** with 800 input and 400 output tokens per request.
This is typical for an internal enterprise assistant.

In [ ]:
calc = CostCalculator()

# Show workload parameters
print("=== Workload Profile ===")
print(f"  Requests/day:      {calc.requests_per_day:,}")
print(f"  Avg input tokens:  {calc.avg_input_tokens:,}")
print(f"  Avg output tokens: {calc.avg_output_tokens:,}")
print(f"  Baseline model:    {calc.baseline_model}")
print(f"  Cache hit rate:    {calc.cache_hit_rate:.0%}")
print(f"  Batch discount:    {calc.batch_discount:.0%}")
print(f"  Batch eligible:    {calc.batch_eligible_pct:.0%}")
print(f"  Tier distribution: {calc.tier_distribution}")
print()

# Run comparison
rows = calc.compare_all()
print(tabulate(
    rows,
    headers=["Scenario", "Daily Cost", "Monthly Cost", "Savings vs Baseline", "Savings %"],
    tablefmt="github",
))

## 4. Scenario Breakdown

Let's visualize how each optimization contributes to savings, using a simple text chart.

In [ ]:
baseline_monthly = calc.scenario_baseline() * 30

scenarios = {
    "Baseline":          calc.scenario_baseline() * 30,
    "+ Caching":         calc.scenario_with_caching() * 30,
    "+ Tiering":         calc.scenario_with_tiering() * 30,
    "+ Batching":        calc.scenario_with_batching() * 30,
    "All combined":      calc.scenario_all_optimized() * 30,
}

# Text-based bar chart
max_cost = max(scenarios.values())
bar_width = 50

print("\n=== Monthly Cost Comparison (bar chart) ===\n")
for name, cost in scenarios.items():
    bar_len = int((cost / max_cost) * bar_width)
    bar = "#" * bar_len
    pct_of_baseline = (cost / baseline_monthly) * 100
    print(f"  {name:<16s} |{bar:<{bar_width}s}| ${cost:>10,.2f}  ({pct_of_baseline:.0f}%)")

## 5. "What-If" Section: Adjust Parameters

Try changing the parameters below and re-running to see how your cost profile changes.
This is where the calculator becomes a planning tool for your own workload.

In [ ]:
# =============================================
# ADJUST THESE PARAMETERS TO MODEL YOUR WORKLOAD
# =============================================

custom_calc = CostCalculator(
    requests_per_day=25_000,       # <-- your daily volume
    avg_input_tokens=1200,         # <-- your average prompt size
    avg_output_tokens=600,         # <-- your average response size
    baseline_model="gpt-4o",       # <-- your current model
    cache_hit_rate=0.40,           # <-- how much repetition in your traffic?
    batch_discount=0.50,           # <-- your provider's batch discount
    batch_eligible_pct=0.30,       # <-- what % of requests can wait?
    tier_distribution={
        "simple": 0.55,            # <-- % simple requests
        "medium": 0.30,            # <-- % medium requests
        "complex": 0.15,           # <-- % complex requests
    },
    tier_model_map={
        "simple": "gpt-4o-mini",
        "medium": "claude-haiku",
        "complex": "gpt-4o",
    },
)

print("=== What-If Analysis ===")
print(f"Workload: {custom_calc.requests_per_day:,} req/day, "
      f"{custom_calc.avg_input_tokens} in / {custom_calc.avg_output_tokens} out tokens\n")

custom_rows = custom_calc.compare_all()
print(tabulate(
    custom_rows,
    headers=["Scenario", "Daily Cost", "Monthly Cost", "Savings vs Baseline", "Savings %"],
    tablefmt="github",
))

## 6. Book Case Study: $45K to $8K

The book describes a real enterprise that reduced their monthly AI spend from **$45,000** to
**$8,000** — an 82% reduction. Let's replicate those numbers with the calculator to see
what parameter combination produces that result.

In [ ]:
# Reverse-engineer the case study parameters
# Target: ~$45K/month baseline, ~$8K/month optimized (82% reduction)

case_study = CostCalculator(
    requests_per_day=50_000,        # high-volume enterprise workload
    avg_input_tokens=1000,
    avg_output_tokens=500,
    baseline_model="gpt-4o",
    cache_hit_rate=0.35,            # semantic caching on common queries
    batch_discount=0.50,
    batch_eligible_pct=0.50,        # half the workload is async reports/summaries
    tier_distribution={
        "simple": 0.60,             # most requests are simple lookups
        "medium": 0.30,
        "complex": 0.10,
    },
    tier_model_map={
        "simple": "gpt-4o-mini",
        "medium": "claude-haiku",
        "complex": "gpt-4o",
    },
)

baseline_monthly = case_study.scenario_baseline() * 30
optimized_monthly = case_study.scenario_all_optimized() * 30
reduction_pct = ((baseline_monthly - optimized_monthly) / baseline_monthly) * 100

print("=== Case Study Replication ===")
print(f"  Baseline monthly cost:   ${baseline_monthly:>10,.2f}")
print(f"  Optimized monthly cost:  ${optimized_monthly:>10,.2f}")
print(f"  Monthly savings:         ${baseline_monthly - optimized_monthly:>10,.2f}")
print(f"  Reduction:               {reduction_pct:.0f}%")
print()

# Full comparison table
case_rows = case_study.compare_all()
print(tabulate(
    case_rows,
    headers=["Scenario", "Daily Cost", "Monthly Cost", "Savings vs Baseline", "Savings %"],
    tablefmt="github",
))

## 7. Annual Projection

For executive presentations, annual numbers are more impactful. Let's project the
case study savings across a full year.

In [ ]:
annual_baseline = baseline_monthly * 12
annual_optimized = optimized_monthly * 12
annual_savings = annual_baseline - annual_optimized

annual_rows = [
    ["Baseline (no optimization)", f"${annual_baseline:>12,.2f}"],
    ["Fully optimized", f"${annual_optimized:>12,.2f}"],
    ["Annual savings", f"${annual_savings:>12,.2f}"],
    ["Reduction", f"{reduction_pct:.0f}%"],
]

print("=== Annual Projection ===")
print(tabulate(annual_rows, tablefmt="github"))

print(f"\nOver three years, these optimizations save approximately "
      f"${annual_savings * 3:,.0f}.")
print("That's real budget that can be redirected to new AI initiatives.")

## 8. Build Your Own Cost Model

Use the cell below as a template. Fill in your actual workload numbers and run it
to get a concrete savings estimate you can take to your leadership team.

In [ ]:
# =============================================
# YOUR WORKLOAD — fill in real numbers
# =============================================

my_calc = CostCalculator(
    requests_per_day=0,            # TODO: your daily request volume
    avg_input_tokens=0,            # TODO: average prompt tokens
    avg_output_tokens=0,           # TODO: average response tokens
    baseline_model="gpt-4o",       # TODO: your current model
    cache_hit_rate=0.0,            # TODO: estimated cache hit rate
    batch_discount=0.50,           # TODO: your provider's batch discount
    batch_eligible_pct=0.0,        # TODO: % of async-eligible requests
    tier_distribution={
        "simple": 0.0,             # TODO: % simple requests
        "medium": 0.0,             # TODO: % medium requests
        "complex": 0.0,            # TODO: % complex requests
    },
)

if my_calc.requests_per_day > 0:
    my_rows = my_calc.compare_all()
    print(tabulate(
        my_rows,
        headers=["Scenario", "Daily Cost", "Monthly Cost", "Savings vs Baseline", "Savings %"],
        tablefmt="github",
    ))
else:
    print("Fill in your workload parameters above and re-run this cell.")

## Key Takeaways

1. **Model your costs before you deploy.** Use a calculator like this one to forecast
   spend and justify optimization investments.

2. **The three levers are caching, tiering, and batching.** Each is independent and
   they stack multiplicatively.

3. **Small percentages matter at scale.** A 30% cache hit rate on 50K daily requests
   eliminates 15,000 LLM calls every single day.

4. **The book's case study ($45K to $8K) is achievable.** A combination of 35% caching,
   60/30/10 tiering, and 50% batch eligibility yields an ~82% cost reduction.

5. **Frame savings in annual terms for executives.** Monthly savings of $37K is compelling;
   annual savings of $444K is a budget line item.

---

*From "AI Enterprise Architect" — Chapter 12: AI Cost Management*